# Stripe - Subscription Churn Impact on Recurring Revenue

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_subscription = pd.read_csv('../Data/010/fct_subscriptions.csv', parse_dates=['start_date','end_date'])

pl_subscription = pl.read_csv('../Data/010/fct_subscriptions.csv', try_parse_dates=True)

# Pregunta 1

### Identifica los primeros 3 niveles (tiers) de suscripción en orden alfabético. No olvides eliminar los valores duplicados. Esta consulta nos ayudará a entender qué valores existen en la columna tier_name.

```SQL
SELECT
    DISTINCT tier_name
FROM fct_subscriptions
ORDER BY tier_name
LIMIT 3;
```

In [34]:
res = df_subscription[['tier_name']].drop_duplicates()
res = res.sort_values('tier_name').head(3).reset_index(drop=True)

In [33]:
res = (
    pl_subscription
    .select('tier_name')
    .unique()
    .sort('tier_name', descending=False)
    .head(3)
)

# Pregunta 2

### Determina cuántos clientes cancelaron sus suscripciones en agosto de 2024 para los niveles (tiers) etiquetados como 'Basic' o 'Premium'. Esta consulta se utiliza para evaluar las tendencias de cancelación en estos niveles de suscripción específicos.

```SQL
SELECT
    COUNT(*) total_customers
FROM fct_subscriptions
WHERE tier_name in ('Basic', 'Premium') AND
      ((EXTRACT(MONTH FROM end_date) = 8) AND
       (EXTRACT(YEAR FROM end_date) = 2024));
```

In [ ]:
end = df_subscription[
    (df_subscription['end_date'].dt.month == 8) &
    (df_subscription['end_date'].dt.year == 2024) &
    (df_subscription['tier_name'].isin(['Basic','Premium']))
].reset_index()

res = end.shape[0]

res

4

In [40]:
res = pl_subscription.filter(
    (pl.col('end_date').dt.month() == 8) &
    (pl.col('end_date').dt.year() == 2024) &
    (pl.col('tier_name').is_in(['Basic','Premium']))
).height

res

4